In [1]:
import tensorflow as tf
import numpy as np
from tensorflow import keras

tf.random.set_seed(42)
np.random.seed(42)

2026-05-12 09:54:09.019380: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1778579649.528687      23 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1778579649.635566      23 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1778579650.733287      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778579650.733337      23 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1778579650.733341      23 computation_placer.cc:177] computation placer alr

In [2]:
import os, sys, time, pickle, glob
from pathlib import Path

IS_KAGGLE = 'KAGGLE_KERNEL_RUN_TYPE' in os.environ

if IS_KAGGLE:
    print('Kaggle GPU')
    os.environ['TF_USE_LEGACY_KERAS'] = '1'
    os.environ['CNN_EPOCHS'] = '20'
    !pip install -q tf-keras
    
    SRC_PATH = '/kaggle/input/cnn-src/src'
    if not os.path.exists(SRC_PATH):
        SRC_PATH = '/kaggle/input/datasets/kefaskurniajonathan/cnn-src-2/src'
    
    sys.path.insert(0, SRC_PATH)
    print(f'Added {SRC_PATH} to sys.path')
    
    os.environ['CNN_DATA_DIR'] = '/kaggle/input/datasets/kefaskurniajonathan/data-intel-cnn'
    
    REPO_ROOT = Path('/kaggle/working')
    os.chdir(REPO_ROOT)
else:
    def _find_root(marker='requirements.txt'):
        p = Path(os.getcwd())
        while p != p.parent:
            if (p / marker).exists(): return p
            p = p.parent
        return p
    REPO_ROOT = _find_root()
    os.chdir(REPO_ROOT)
    sys.path.insert(0, str(REPO_ROOT / 'src'))
    print('Running Locally. Root:', REPO_ROOT)

Kaggle GPU
Added /kaggle/input/datasets/kefaskurniajonathan/cnn-src-2/src to sys.path


In [3]:
from cnn.train_keras import (
    build_conv2d_model, build_locally_connected_model,
    get_data_loaders, train, IMG_SIZE
)

MODELS_DIR  = "models/cnn"
HISTORY_DIR = "models/cnn/history"

os.makedirs(MODELS_DIR,  exist_ok=True)
os.makedirs(HISTORY_DIR, exist_ok=True)

In [4]:
print("Loading data...")
train_ds, val_ds = get_data_loaders(img_size=IMG_SIZE)
train_ds_lc, val_ds_lc = get_data_loaders(img_size=(64, 64))

Loading data...
Found 14034 files belonging to 6 classes.


I0000 00:00:1778579714.613861      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13757 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1778579714.619658      23 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13757 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


Found 3000 files belonging to 6 classes.
Found 14034 files belonging to 6 classes.
Found 3000 files belonging to 6 classes.


In [5]:
FILTER_CONFIGS = {
    "base":  {2: [32, 64],       3: [32, 64, 128]},
    "large": {2: [64, 128],      3: [64, 128, 256]},
}

VARIATIONS = []
for n_layers in [2, 3]:
    for f_type in ["base", "large"]:
        for k_size in [3, 5]:
            for p_type in ["max", "avg"]:
                name = f"cnn-{n_layers}L-{f_type}-k{k_size}-{p_type}"
                VARIATIONS.append({
                    "name": name,
                    "num_conv_layers": n_layers,
                    "filters": FILTER_CONFIGS[f_type][n_layers],
                    "kernel_sizes": [k_size] * n_layers,
                    "pooling": p_type,
                })
print(f"Total arsitektur: {len(VARIATIONS)}")

Total arsitektur: 16


In [6]:
total_variations = len(VARIATIONS)
histories = {}
for i, v in enumerate(VARIATIONS):
    name = v['name']
    print(f'\n' + '='*50)
    print(f'[ MODEL {i+1} dari {total_variations+1} ]')
    print(f'Melatih: {name}')
    print('='*50 + '\n')
    
    kwargs = {k: val for k, val in v.items() if k != 'name'}
    model = build_conv2d_model(**kwargs)
    history = train(model, train_ds, val_ds, model_name=name)
    
    histories[name] = history.history
    with open(os.path.join(HISTORY_DIR, f'{name}_history.pkl'), 'wb') as f:
        pickle.dump(history.history, f)

with open(os.path.join(MODELS_DIR, 'all_conv2d_histories.pkl'), 'wb') as f:
    pickle.dump(histories, f)
print("\nOK Semua Conv2D selesai.")


[ MODEL 1 dari 17 ]
Melatih: cnn-2L-base-k3-max

Epoch 1/20


I0000 00:00:1778579729.539596      81 cuda_dnn.cc:529] Loaded cuDNN version 91002
I0000 00:00:1778579731.226235      83 service.cc:152] XLA service 0x7e890966ee60 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1778579731.226287      83 service.cc:160]   StreamExecutor device (0): Tesla T4, Compute Capability 7.5
I0000 00:00:1778579731.226295      83 service.cc:160]   StreamExecutor device (1): Tesla T4, Compute Capability 7.5
I0000 00:00:1778579731.627321      83 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


439/439 [==============================] - 32s 61ms/step - loss: 1.3263 - accuracy: 0.4315 - val_loss: 1.1784 - val_accuracy: 0.5307
Epoch 2/20
439/439 [==============================] - 11s 25ms/step - loss: 1.0709 - accuracy: 0.5708 - val_loss: 1.0245 - val_accuracy: 0.6140
Epoch 3/20
439/439 [==============================] - 11s 26ms/step - loss: 0.9452 - accuracy: 0.6259 - val_loss: 0.9509 - val_accuracy: 0.6240
Epoch 4/20
439/439 [==============================] - 11s 25ms/step - loss: 0.8734 - accuracy: 0.6603 - val_loss: 0.8430 - val_accuracy: 0.6957
Epoch 5/20
439/439 [==============================] - 12s 27ms/step - loss: 0.8175 - accuracy: 0.6900 - val_loss: 0.8317 - val_accuracy: 0.6710
Epoch 6/20
439/439 [==============================] - 13s 28ms/step - loss: 0.7816 - accuracy: 0.7057 - val_loss: 0.8242 - val_accuracy: 0.6803
Epoch 7/20
439/439 [==============================] - 11s 25ms/step - loss: 0.7474 - accuracy: 0.7200 - val_loss: 0.7494 - val_accuracy: 0.7250
Epo

In [7]:
print("\nTraining LocallyConnected2D...")
lc_model = build_locally_connected_model()
lc_history = train(lc_model, train_ds_lc, val_ds_lc, model_name="lc-model")
with open(os.path.join(HISTORY_DIR, "lc-model_history.pkl"), "wb") as f:
    pickle.dump(lc_history.history, f)
print("\nOK Training Selesai!")


Training LocallyConnected2D...
Epoch 1/20
439/439 [==============================] - 235s 298ms/step - loss: 1.3721 - accuracy: 0.4577 - val_loss: 1.2581 - val_accuracy: 0.5337
Epoch 2/20
439/439 [==============================] - 51s 115ms/step - loss: 1.1793 - accuracy: 0.5442 - val_loss: 1.1895 - val_accuracy: 0.5377
Epoch 3/20
439/439 [==============================] - 50s 113ms/step - loss: 1.1037 - accuracy: 0.5777 - val_loss: 1.0928 - val_accuracy: 0.5703
Epoch 4/20
439/439 [==============================] - 50s 114ms/step - loss: 1.0554 - accuracy: 0.5916 - val_loss: 1.0848 - val_accuracy: 0.5780
Epoch 5/20
439/439 [==============================] - 50s 113ms/step - loss: 1.0188 - accuracy: 0.6080 - val_loss: 1.0308 - val_accuracy: 0.5927
Epoch 6/20
439/439 [==============================] - 49s 112ms/step - loss: 0.9617 - accuracy: 0.6313 - val_loss: 1.0474 - val_accuracy: 0.5997
Epoch 7/20
439/439 [==============================] - 50s 115ms/step - loss: 0.9316 - accuracy: 0